In [19]:
import requests
import pandas as pd

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType, TimestampType
import time
from pyspark.sql import functions as F

from datetime import datetime
import os
from pathlib import Path


%load_ext sparksql_magic


home_dir = os.path.expanduser("~")
delta_base_path = os.path.join(home_dir, "lakehouse_test/sport_games")

In [ ]:
# 1. Configure Spark Session for Delta Lake
builder = SparkSession.builder \
    .appName("DeltaLakeLocal") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Automatically downloads the correct matching Delta jars
spark = configure_spark_with_delta_pip(builder).getOrCreate()
%config SparkSqlMagic.spark = spark

your 131072x1 screen size is bogus. expect trouble
26/05/19 21:19:06 WARN Utils: Your hostname, DESKTOP-IS86RQE resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/19 21:19:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/obaliuta/miniconda3/envs/spark_with_delta/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/obaliuta/.ivy2/cache
The jars for the packages stored in: /home/obaliuta/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-21701bff-cef0-4724-93cd-34996d917413;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 466ms :: artifacts dl 18ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |

In [3]:
last_days = 1 #yesterday
stamp = pd.Timestamp(datetime.now()).tz_localize('UTC') - pd.Timedelta(days=last_days)
string_stamp = stamp.strftime('%Y-%m-%d')
KEY = os.environ.get('SPORT_KEY')
if not KEY:
    key_file = Path.home() / '.config' / 'sport_games' / 'key'
    if key_file.exists():
        KEY = key_file.read_text().strip()
    else:
        raise FileNotFoundError("API key not found. Set SPORT_KEY or run scripts/store_key.sh <key>")

26/05/19 21:19:26 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [37]:
response = requests.get(
    f"https://therundown.io/api/v2/sports/4/events/{string_stamp}?key={KEY}"
)

if response.status_code == 200:
    resp = response.json()
    rows = []
    
    # Navigate: events -> markets -> participants -> lines -> prices
    events = resp.get('events', [])
    
    for event in events:
        event_id = event.get('event_id', '')
        event_date = event.get('event_date', '')
        
        # Iterate through markets (e.g., moneyline, spread, totals)
        for market in event.get('markets', []):
            market_id = market.get('market_id', '')
            market_desc = market.get('market_description', '')
            
            # Iterate through participants (teams)
            for participant in market.get('participants', []):
                team_id = participant.get('id', '')
                team_type = participant.get('type', '')
                team_name = participant.get('name', '')
                
                # Iterate through lines
                for line in participant.get('lines', []):
                    line_id = line.get('id', '')
                    line_value = line.get('value', '')  # May not exist for moneyline
                    
                    # Iterate through prices (dict keyed by bookmaker/market)
                    prices = line.get('prices', {})
                    for market_key, price_info in prices.items():
                        row = {
                            'event_id': event_id,
                            'event_date': event_date,
                            'market_id': market_id,
                            'market_description': market_desc,
                            'team_id': team_id,
                            'team_type': team_type,
                            'team_name': team_name,
                            'line_id': line_id,
                            'line_value': line_value,
                            'market_key': market_key,
                            'price_id': price_info.get('id', ''),
                            'price': price_info.get('price'),
                            'price_delta': price_info.get('price_delta'),
                            'is_main_line': price_info.get('is_main_line', False),
                            'updated_at': price_info.get('updated_at'),
                            'closed_at': price_info.get('closed_at')
                        }
                        rows.append(row)
    
    # Create DataFrame
    data = pd.DataFrame(rows)
    
    # Convert datetime columns
    for col in ['event_date', 'updated_at', 'closed_at']:
        if col in data.columns:
            data[col] = pd.to_datetime(data[col], errors='coerce')
    
else:
    print(f"Error: {response.status_code}")
    data = pd.DataFrame()


In [ ]:
try:
    #getting last id
    last_id_val = spark.read.table(f"FROM delta.`{delta_base_path}/bronze/dataset`").select(F.spark_max("id")).collect()[0][0]
    if last_id_val is None:
        last_id_val = 0
except:
    # 0 if not found
    last_id_val = 0

data_spark = (spark.createDataFrame(data)
              .withColumn("Timestamp", F.current_timestamp())
              .withColumn("id", F.monotonically_increasing_id() + F.lit(last_id_val + 1)))  

# 3. Save DataFrame as a Delta Table

data_spark.write.format("delta").option("mergeSchema", "true").mode("append").save(delta_base_path+"/bronze/dataset")
print(f"Data is written successfully at: {delta_base_path}/bronze/dataset")

Delta table created successfully at: /home/obaliuta/lakehouse_test/sport_games/bronze/dataset


In [8]:
bronze = spark.sql(f"SELECT * FROM delta.`{delta_base_path}/bronze/dataset`")

bronze = bronze.drop("is_main_line")

bronze.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_date: timestamp (nullable = true)
 |-- market_id: long (nullable = true)
 |-- market_description: string (nullable = true)
 |-- team_id: long (nullable = true)
 |-- team_type: string (nullable = true)
 |-- team_name: string (nullable = true)
 |-- line_id: string (nullable = true)
 |-- line_value: string (nullable = true)
 |-- market_key: string (nullable = true)
 |-- price_id: string (nullable = true)
 |-- price: long (nullable = true)
 |-- price_delta: double (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- closed_at: timestamp (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- id: long (nullable = true)



In [ ]:
#normalization
event = (bronze.filter(F.col("team_type") == "TYPE_TEAM")
          .select("event_id", "event_date", "team_name").distinct()
          .groupBy("event_id", "event_date")
          .agg(F.collect_set("team_name").alias("teams"))
)

market = bronze.select("market_id", "market_description").distinct()

team = bronze.select("team_id", "team_name", "team_type").distinct()

event_lines = bronze.drop("event_date", "market_description", "team_name", "team_type") 

In [15]:
#writing to the lakehouse
event.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(delta_base_path+"/silver/event")

market.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(delta_base_path+"/silver/market")

team.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(delta_base_path+"/silver/team")

event_lines.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(delta_base_path+"/silver/event_lines")

In [22]:
%%sparksql
SELECT * FROM delta.`/home/obaliuta/lakehouse_test/sport_games/silver/event_lines/` ORDER BY id DESC LIMIT 10   

event_id,market_id,team_id,line_id,line_value,market_key,price_id,price,price_delta,updated_at,closed_at,Timestamp,id
f96ec16dfe9aea4c2257944b936e1bac,3,10,54aaec0128952a2ca881e47c149cb767,241.5,19,203969813,-940,20.0,2026-05-10 23:43:19,2026-05-10 23:43:19,2026-05-19 13:44:17.670500,42949673211
f96ec16dfe9aea4c2257944b936e1bac,3,10,5b51a300c9c02d9c28d0b4e3acd73788,240.5,19,203955558,-840,20.0,2026-05-10 23:43:19,2026-05-10 23:43:19,2026-05-19 13:44:17.670500,42949673210
f96ec16dfe9aea4c2257944b936e1bac,3,10,d7113acbe6248c7e39ab5e6c6b73e5a8,"226, 5",22,203958379,-220,nan,2026-05-09 16:15:31,2026-05-09 16:15:31,2026-05-19 13:44:17.670500,42949673209
f96ec16dfe9aea4c2257944b936e1bac,3,10,f0763535429507cf9d6461d71afc9f97,239.5,19,203935573,-750,30.0,2026-05-10 23:43:19,2026-05-10 23:43:19,2026-05-19 13:44:17.670500,42949673208
f96ec16dfe9aea4c2257944b936e1bac,3,10,f0763535429507cf9d6461d71afc9f97,239.5,23,203922220,-1200,100.0,2026-05-10 23:43:06,2026-05-10 23:43:06,2026-05-19 13:44:17.670500,42949673207
f96ec16dfe9aea4c2257944b936e1bac,3,10,beb1e740ca72b41933f7ff3040f2359e,238.5,19,203935571,-670,20.0,2026-05-10 23:43:19,2026-05-10 23:43:19,2026-05-19 13:44:17.670500,42949673206
f96ec16dfe9aea4c2257944b936e1bac,3,10,beb1e740ca72b41933f7ff3040f2359e,238.5,23,203922239,-1050,50.0,2026-05-10 23:43:06,2026-05-10 23:43:06,2026-05-19 13:44:17.670500,42949673205
f96ec16dfe9aea4c2257944b936e1bac,3,10,cfcc7ceddf61ff98ce7a0370d45aa990,237.5,19,203935569,-620,10.0,2026-05-10 23:43:19,2026-05-10 23:43:19,2026-05-19 13:44:17.670500,42949673204
f96ec16dfe9aea4c2257944b936e1bac,3,10,cfcc7ceddf61ff98ce7a0370d45aa990,237.5,23,203922261,-900,-50.0,2026-05-10 23:43:06,2026-05-10 23:43:06,2026-05-19 13:44:17.670500,42949673203
f96ec16dfe9aea4c2257944b936e1bac,3,10,ff0a1cfedf55fbe36f9173a27140843f,236.5,19,203935567,-547,17.0,2026-05-10 23:43:19,2026-05-10 23:43:19,2026-05-19 13:44:17.670500,42949673202


In [ ]:
spark.stop()